In [1]:
import numpy as np
import json

from posture import gen_posture
from lib import inverse_kinematics
from motion import gen_walk_path

servo_min = 125
servo_max = 575
servo_range = servo_max - servo_min

In [2]:
with open("config.json", "r", encoding="utf-8") as read_file:
    config =  json.load(read_file)


In [3]:
standby = gen_posture(60, 75, config)
angles = inverse_kinematics(standby, config)

np.round(angles/180 * servo_range +servo_min)

array([[350., 425., 387.],
       [350., 275., 312.],
       [350., 275., 312.],
       [350., 275., 312.],
       [350., 425., 388.],
       [350., 425., 388.]])

In [4]:
standby

array([[ 1.07683676e+02,  1.33683676e+02, -6.47350133e+01],
       [ 1.47565695e+02,  0.00000000e+00, -6.47350133e+01],
       [ 1.07683676e+02, -1.33683676e+02, -6.47350133e+01],
       [-1.07683676e+02,  1.33683676e+02, -6.47350133e+01],
       [-1.47565695e+02, -1.35563128e-14, -6.47350133e+01],
       [-1.07683676e+02, -1.33683676e+02, -6.47350133e+01]])

In [13]:
path_walk = gen_walk_path(standby)



In [16]:
np.shape(path_walk)[0]

28

In [27]:
fp = open("./motion.h", "w")

fp.write("/*\n" + " * This is an automatically generated code\n" + " *\n")

fp.write(" */\n\n")

fp.write("#ifndef MOTION_H\n")
fp.write("#define MOTION_H\n\n")

fp.write("static int path_walk_length = "+str(np.shape(path_walk)[0])+";\n")
fp.write("static int path_walk["+str(np.shape(path_walk)[0])+"][6][3] = {")

for idx in range(0, np.shape(path_walk)[0]):
    angles = inverse_kinematics(path_walk[idx,:,:], config)
    path_walk_pwm = np.round(angles/180 * servo_range +servo_min)
    path_walk_pwm = path_walk_pwm.astype(int)

    fp.write("{")
    for r in range(0, 5):
        fp.write("{"+str(path_walk_pwm[r, 0])+", " +str(path_walk_pwm[r, 1])+", "+str(path_walk_pwm[r, 2])+"},\n")
    fp.write("{"+str(path_walk_pwm[5, 0])+", " +str(path_walk_pwm[5, 1])+", "+str(path_walk_pwm[5, 2])+"}}")

    if idx == (np.shape(path_walk)[0]-1):
        fp.write("};\n\n")
    else:
        fp.write(",\n")
    
# fp.write("}\n")

fp.write("#endif // MOTION_H\n")

fp.close()